In [13]:
import os
from pathlib import Path

import contextily as cx
import geopandas as gpd
import matplotlib.colors as mcol
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
pg_path = Path(os.getenv("PG_PATH"))
ghsl_path = Path(os.getenv("GHSL_PATH"))
data_path = Path(os.getenv("DATA_PATH"))

In [47]:
df = pd.read_csv(data_path / "generated/census_pop/WORLDPOP.csv", index_col=0)
df.columns = ["pop"]
df

,pop
index,
0,133.086386
1,360.998568
2,171.970004
3,59.205567
4,627.912185
...,...
64060,1410.327053
64061,1531.186970
64062,1764.568747


,0
0,31.705072
1,213.563965
2,425.964935
3,151.993240
4,461.308990
...,...
64060,1285.260742
64061,1785.111938
64062,1787.849487
64063,1160.559448


In [ ]:
fpath = data_path / "pop_ghsl.csv"

if fpath.exists():
    pop_ghsl = pd.read_csv(fpath, index_col="Unnamed: 0")["0"]
else:
    pop_ghsl = calculate_mesh_pop(boxes_all)
    pop_ghsl.to_csv(fpath)

In [6]:
fpath = data_path / "pop_census.csv"

if fpath.exists():
    pop_census = pd.read_csv(fpath, index_col="index")["pop_frac"]
else:
    pop_census = calculate_pop_census(df_all, boxes_all)
    pop_census.to_csv(fpath)

In [8]:
fpath = data_path / "boxes_pop.gpkg"

if fpath.exists():
    boxes_pop = gpd.read_file(fpath)
else:
    boxes_pop = merge_pops(boxes_all, pop_census=pop_census, pop_ghsl=pop_ghsl)
    boxes_pop.to_file(fpath)

In [23]:
mun_list = (
    gpd.read_file(pg_path / "initial/metropoli/2020/Metropolis_2020.shp")
    .query("TIPO_MET.isin(['Zona metropolitana', 'Metrópoli municipal'])")
    .dissolve("CVE_MET")[["geometry"]]
    .reset_index()
    .to_crs("ESRI:54009")
)

In [ ]:
figure_path = data_path / "figures/blocks_inside_ghsl"
df_path = data_path / "polygons/blocks_inside_ghsl"

for cve in mun_list["CVE_MET"]:
    temp = boxes_pop.sjoin(
        mun_list.query(f"CVE_MET == '{cve}'"), how="inner", predicate="intersects"
    )
    temp.to_file(df_path / f"{cve}.gpkg")

    norm = mcol.TwoSlopeNorm(0, temp["difference"].min(), temp["difference"].max())

    fig, ax = plt.subplots(figsize=(10, 10))
    temp.plot(
        column="difference",
        norm=norm,
        ax=ax,
        legend=True,
        cmap="RdBu",
        lw=0.3,
        ec="gray",
    )
    ax.axis("off")
    cx.add_basemap(ax=ax, crs="ESRI:54009", source=cx.providers.CartoDB.Positron)
    ax.set_title(f"{cve}\ncenso - GHSL")

    fig.savefig(figure_path / f"{cve}.jpg", bbox_inches="tight", dpi=150)
    plt.close()